Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left"> <td>      <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Convert_Gemma_3_270M_to_LiteRT_for_MediaPipe_LLM_Inference_API.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

# 將 Gemma 3 270M 轉換為 LiteRT 以與 MediaPipe LLM Inference API 一起使用

此notebook 轉換Gemma 3 270M 以與[MediaPipe LLM Inference API](https://ai.google.dev/edge/mediapipe/solutions/genai/llm_inference) 一起使用，library 可在行動裝置或網頁瀏覽器中啟用inference。整個過程大約需要15分鐘：
1. 設定Colab環境
2. 從Hugging Face載入模型
3. 使用 AI Edge Torch 轉換器轉換模型
4. 使用 MediaPipe 任務捆綁器打包模型
5. 下載模型

Gemma 3 270M 專為特定任務fine-tuning 而設計，旨在在行動、網路和邊緣設備上實現高效效能。您可以使用此 [notebook](https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Demos/Emoji-Gemma-on-Web/resources/Fine_tune_Gemma_3_270M_for_emoji_generation.ipynb) 微調您自己的模型，並在轉換後在演示 [網頁應用程式](https://github.com/google-gemini/gemma-cookbook/tree/main/Demos/Emoji-Gemma-on-Web/app-mediapipe) 中執行它。
## 設定開發環境

第一步是使用 pip 安裝軟體包。

In [ ]:
%pip uninstall -y tensorflow
%pip install -U tf-nightly==2.21.0.dev20250819 ai-edge-torch==0.6.0 protobuf transformers
%pip install -U jax jaxlib

重新啟動會話 runtime 以確保您正在使用新安裝的軟體包。

## 載入模型
若要存取 Hugging Face 上的模型，請提供您的[存取權杖](https://huggingface.co/settings/tokens)。您可以將其儲存為Colab secret 在左側工具列中，方法是將`HF_TOKEN` 指定為「名稱」並新增您唯一的token 作為「值」。

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

指定要轉換的模型的 Hugging Face 儲存庫 ID。它將保存到您的 Colab 文件中進行轉換。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_author = ""                                         #@param {type:"string"}
model_name = "myemoji-gemma-3-270m-it"                    #@param {type:"string"}

repo_id = f"{model_author}/{model_name}"                  # Model to convert
save_path = f"/content/{model_name}"                      # Path to save resized model

model = AutoModelForCausalLM.from_pretrained(repo_id)     # Load the model
tokenizer = AutoTokenizer.from_pretrained(repo_id)        # Load the tokenizer

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved to {save_path}")

## 轉換模型
使用 [AI Edge Torch](https://github.com/google-ai-edge/ai-edge-torch) 轉換器對模型進行轉換和量化。您可以根據任務要求調整轉換參數：
* `prefill_seq_len`：支援輸入的最大長度
* `kv_cache_max_len`：預先填入的最大值+解碼上下文長度
* `quantize`：量化方案。 8 位元整數量化 (INT8) 適合 Web 環境

這大約需要 10 分鐘。 .tflite 模型將暫時儲存到您的Colab 檔案。

In [ ]:
from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache

# Get model, set export settings, and convert to .tflite
pytorch_model = gemma3.build_model_270m(save_path)
export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True
converter.convert_to_tflite(
    pytorch_model,
    output_path="/content",
    output_name_prefix=model_name,
    prefill_seq_len=128,
    kv_cache_max_len=512,
    quantize="dynamic_int8",
    export_config=export_config,
)

print (f"Model converted to .tflite and saved to {save_path}")

## 建立 MediaPipe 任務包

MediaPipe 任務檔案 (.task) 捆綁了原始模型 tokenizer、LiteRT 模型 (.tflite) 以及執行端對端 inference 與 MediaPipe LLM Inference API 所需的附加元資料。
要使用捆綁程序，請在此步驟中安裝 MediaPipe PyPI 套件 (>0.10.14)，因為它帶有自己的一組依賴項。

In [ ]:
%pip install mediapipe

全新安裝 `protobuf` 和 `tensorflow` 並重新啟動 Colab runtime 以獲取最新版本。

In [ ]:
%pip uninstall protobuf -y && pip install protobuf
%pip uninstall tensorflow -y -q && pip install tensorflow

現在，您將設定並建立任務包：
1. 更新 `tflite_model` 以指向 Colab 檔案中新轉換的 .tflite 模型
2. 更新`tokenizer_model`以指向從Hugging Face Hub下載的tokenizer.model。
3. 將您的.task 檔案命名為`output_filename`。

In [ ]:
from mediapipe.tasks.python.genai import bundler

config = bundler.BundleConfig(
    tflite_model="/content/myemoji-gemma-3-270m-it_q8_ekv512.tflite",     # Point to your converted .tflite model
    tokenizer_model="/content/myemoji-gemma-3-270m-it/tokenizer.model",   # Point to the downloaded model's tokenizer.model file
    start_token="<bos>",
    stop_tokens=["<eos>", "<end_of_turn>"],
    output_filename="/content/myemoji-gemma-3-270m-it.task",              # Specify the final model filename
    prompt_prefix="<start_of_turn>user\n",
    prompt_suffix="<end_of_turn>\n<start_of_turn>model\n",
)
bundler.create_bundle(config)

print(f"Model .task bundle saved to {config.output_filename}")

## 在設備上下載並執行您的模型

您的模型現在已準備好使用MediaPipe LLM Inference API在設備上inference！
從 Colab 環境下載 .task 檔案以在專案中使用它。

In [ ]:
from google.colab import files

files.download(config.output_filename)

在[表情符號生成網頁應用程式](https://github.com/google-gemini/gemma-cookbook/tree/main/Demos/Emoji-Gemma-on-Web/app-mediapipe) 中嘗試一下，該應用程式直接在瀏覽器中執行模型。您也可以探索[文件](https://ai.google.dev/edge/mediapipe/solutions/genai/llm_inference)來建立跨平台行動和網路應用程式。